In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
from jax.scipy.linalg import solve_triangular

import gpjax as gpx
import optax as ox
import paramax as px

from non_parametric_pro import ula

from non_parametric_pro.density import ProParameters, pro_logdensity_fn, pro_score_fn
from non_parametric_pro.ula import parametric_ula
from non_parametric_pro.util import posterior_function_draws
from non_parametric_pro.data.synthetic import make_contaminated_data, make_mixture_data
from non_parametric_pro.gp import _full_gp_basis
from non_parametric_pro.adaptation.parameter_adaptation import parameter_adaptation
from non_parametric_pro.data.kampala_airquality import load_kampala_airquality_records, kampala_forecasting_split, kampala_site_ids
from non_parametric_pro.inducing import PointInducingBasis, compute_inducing_basis, kmeans_inducing_points

from scipy import stats
from blackjax.util import run_inference_algorithm

from non_parametric_pro.util import nlpd_gp, nlpd_pro

jax.config.update("jax_enable_x64", True)

from non_parametric_pro.data.kampala_airquality import load_kampala_airquality_records, kampala_forecasting_split, kampala_site_ids

In [2]:
records = load_kampala_airquality_records()
site_ids = kampala_site_ids(records)
data = kampala_forecasting_split(records, site_ids[0], max_train=100000)

In [3]:
x_train, y_train = jnp.array(data.x_train), jnp.array(data.y_train)
x_test, y_test = jnp.array(data.x_test), jnp.array(data.y_test)

In [8]:
data.y_std*y_test.squeeze() + data.y_mean

Array([21.988 , 39.9906, 30.7024, 21.4573, 21.805 , 22.5391, 45.1565,
       39.7121, 51.6484, 16.1656, 46.9788, 58.2426, 23.2407, 21.1711,
       33.7816, 21.7161, 31.1152, 42.1831, 45.1426, 33.3301, 49.146 ,
       56.2703, 70.5736, 27.1686], dtype=float64)

In [5]:
data._fields

('x_train',
 'y_train',
 'x_test',
 'y_test',
 'y_mean',
 'y_std',
 'site_id',
 'fold')